# 02 - Ablation figures

Reads the tables produced by `01_ablation_tables.ipynb` and draws the figures
for the ablation chapter: which feature matters most, how that changes across
the five missingness patterns, and the fine-grained per-attribute effect.

**Inputs** (from `analysis/tables/`, produced by notebook 01)

* `ranking.csv` - one importance value per (log, excluded feature)
* `delta_aggregated.csv` - Delta_cat / Delta_num per (log, excluded feature, pattern)
* `delta_per_attribute.csv` - the fine-grained Delta_i,j per target attribute

**Outputs** (written as PNG files in `analysis/figures/`)

| file | figure |
|---|---|
| `01_ranking_by_log.png` | attribute importance ranking, one panel per log |
| `02_pattern_heatmap.png` | Delta_cat by (feature, missingness pattern), one panel per log |
| `03_attribute_breakdown_<log>.png` | fine-grained Delta_i,j matrix, one file per log |
| `04_top_feature_by_pattern.png` | the single most important feature of each log, across patterns |

Color follows the sign of Delta everywhere in this notebook, not the chart
type or the log: **orange = removing the feature hurts the reconstruction
(Delta > 0), purple = removing it does not (Delta <= 0)**. Faded bars mark
comparisons where the simple t-test of notebook 01 did not reach `p < 0.05`.

In [2]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm, to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [3]:
root_path = os.getcwd()
root = os.path.dirname(root_path)

In [4]:
tables_dir = os.path.join(root, "analysis", "tables")
figures_dir = os.path.join(root, "analysis", "figures")
os.makedirs(figures_dir, exist_ok=True)


logs = ["bpi_2012_CZ", "bpi_2013_CZ", "sp2020_CZ", "BPI20_RequestForPayment_CZ"]
log_label = {
    "bpi_2012_CZ": "BPI 2012",
    "bpi_2013_CZ": "BPI 2013",
    "sp2020_CZ": "SP 2020",
    "BPI20_RequestForPayment_CZ": "BPI 2020 RfP",
}
patterns = ["odd", "even", "window", "random", "attr_level"]

In [5]:
ranking = pd.read_csv(os.path.join(tables_dir, "ranking.csv"))
delta_aggregated = pd.read_csv(os.path.join(tables_dir, "delta_aggregated.csv"))
delta_per_attribute = pd.read_csv(os.path.join(tables_dir, "delta_per_attribute.csv"))

In [6]:
COLOR_HURT = "#b35806"   # Delta > 0: removing the feature degrades the repair
COLOR_HELP = "#542788"   # Delta <= 0: removing the feature does not hurt it
COLOR_GRID = "#dddddd"
COLOR_AXIS = "#888888"
CMAP_DIVERGING = "PuOr_r"  # low (purple) = helps, high (orange) = hurts

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
})


def style_ax(ax, grid_axis="x"):
    """Thin spines, recessive grid: the data should be the only thing that stands out."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(COLOR_AXIS)
    ax.spines["bottom"].set_color(COLOR_AXIS)
    ax.tick_params(colors=COLOR_AXIS)
    ax.grid(axis=grid_axis, color=COLOR_GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)


def sign_color(values):
    return [COLOR_HURT if v > 0 else COLOR_HELP for v in values]


print(f"root    : {root}")
print(f"figures : {figures_dir}")

root    : /home/danbi/Projects/SANAGRAPH
figures : /home/danbi/Projects/SANAGRAPH/analysis/figures


## 1. Which feature matters most

`delta_cat` averaged over the five missingness patterns (eq. 2.2, then
averaged again as in notebook 01, section 6). Bars are the attributes
ablation actually subtracts information from at constant graph topology.
`Activity` is structurally different (section 2.2.3 of the thesis): removing
it disconnects the graph, so it is drawn as a separate reference line
instead of competing for a rank.

In [9]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for ax, log in zip(axes.flat, logs):
    r = ranking[ranking.log == log]
    others = r[~r.structural].sort_values("delta_cat")
    activity = r[r.structural]

    y = np.arange(len(others))
    alphas = np.where(others.n_significant > 0, 1.0, 0.35)
    colors = [to_rgba(c, alpha=a) for c, a in zip(sign_color(others.delta_cat), alphas)]

    ax.barh(y, others.delta_cat, color=colors, height=0.6, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(others.excluded_feature)
    ax.set_ylim(-0.7, len(others) - 0.3)
    ax.axvline(0, color=COLOR_AXIS, linewidth=0.8)

    if len(activity):
        ax.axvline(activity.delta_cat.iloc[0], color="black", linewidth=1.2,
                   linestyle="--", zorder=2)

    ax.set_title(log_label[log], fontsize=11, fontweight="bold")
    ax.set_xlabel(r"$\Delta_{cat}$  (mean over patterns)")
    style_ax(ax, grid_axis="x")

legend_handles = [
    Patch(facecolor=COLOR_HURT, label="removing it hurts the repair (Δ > 0)"),
    Patch(facecolor=COLOR_HELP, label="removing it does not (Δ ≤ 0)"),
    Line2D([0], [0], color="black", linewidth=1.2, linestyle="--",
           label="Activity (structural, see section 2.2.3 - not ranked)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, -0.03), fontsize=9)
fig.suptitle("Attribute importance ranking (Leave-One-Feature-Out ablation)",
             fontsize=13, y=1.02)
fig.text(0.5, -0.06, "Bar length = value; faded bars did not reach p < 0.05 (simple t-test).",
         ha="center", fontsize=8.5, color=COLOR_AXIS)
fig.tight_layout()

path = os.path.join(figures_dir, "01_ranking_by_log.png")
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"saved {path}")

saved /home/danbi/Projects/SANAGRAPH/analysis/figures/01_ranking_by_log.png


## 2. Does the ranking depend on how events go missing

The four masking strategies (`odd`, `even`, `window`, `random`) plus the
attribute-level pattern (`attr_level`) stress the model differently
(section 1.5.4 and 2.4.1). This heatmap shows `delta_cat` per (feature,
pattern), so a row that stays one color tells a pattern-independent effect,
while a row that changes color tells a pattern-dependent one.

In [10]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, log in zip(axes.flat, logs):
    d = delta_aggregated[(delta_aggregated.log == log) & (~delta_aggregated.structural)]
    order = ranking[(ranking.log == log) & (~ranking.structural)].sort_values("rank").excluded_feature
    matrix = (d.pivot(index="excluded_feature", columns="pattern", values="delta_cat")
              .reindex(index=order, columns=patterns))

    vmax = np.nanmax(np.abs(matrix.to_numpy()))
    norm = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
    im = ax.imshow(matrix.to_numpy(), cmap=CMAP_DIVERGING, norm=norm, aspect="auto")

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.to_numpy()[i, j]
            if np.isnan(value):
                continue
            ax.text(j, i, f"{value:+.3f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(value) > vmax * 0.55 else "#222222")

    ax.set_xticks(range(len(patterns)))
    ax.set_xticklabels(patterns, rotation=30, ha="right")
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_title(log_label[log], fontsize=11, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=r"$\Delta_{cat}$")

fig.suptitle("Feature importance across missingness patterns", fontsize=13, y=1.02)
fig.tight_layout()

path = os.path.join(figures_dir, "02_pattern_heatmap.png")
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"saved {path}")

saved /home/danbi/Projects/SANAGRAPH/analysis/figures/02_pattern_heatmap.png


## 3. Per-attribute breakdown

The aggregated Delta treats every surviving attribute as equally important
(notebook 01, section 5). This figure is the breakdown behind it: for each
ablated feature (row) and each target attribute (column), the exact
Delta_i,j of eq. (2.1), averaged over the five patterns. Categorical targets
(accuracy) and numerical targets (MAE) are kept in two separate panels since
they are not on the same scale. `Activity` is listed last, separated by a
line, for the reason given above.

In [12]:
def attribute_breakdown_figure(log):
    d = (delta_per_attribute[delta_per_attribute.log == log]
         .groupby(["excluded_feature", "structural", "target", "target_kind"], as_index=False)
         .delta.mean())

    order = list(ranking[(ranking.log == log) & (~ranking.structural)]
                 .sort_values("rank").excluded_feature)
    order.append("Activity")  # always last, set apart by the divider line below

    kinds = [("categorical", "accuracy"), ("numerical", "MAE")]
    panels = [(kind, label, d[d.target_kind == kind]) for kind, label in kinds]
    panels = [(kind, label, p) for kind, label, p in panels if len(p)]

    # width follows the number of columns, so labels never crowd (fixed
    # per-panel width was the bug: 11 categorical targets in one 5.5in axis).
    target_lists = [sorted(p.target.unique()) for _, _, p in panels]
    widths = [max(3.0, 0.85 * len(t) + 2.3) for t in target_lists]
    height = 0.42 * len(order) + 2.4

    fig, axes = plt.subplots(1, len(panels), figsize=(sum(widths), height),
                             gridspec_kw={"width_ratios": widths})
    axes = np.atleast_1d(axes)

    for ax, (kind, metric_label, panel), targets in zip(axes, panels, target_lists):
        matrix = panel.pivot(index="excluded_feature", columns="target", values="delta")
        matrix = matrix.reindex(index=order, columns=targets)

        vmax = np.nanmax(np.abs(matrix.to_numpy()))
        vmax = vmax if vmax > 0 else 1.0
        norm = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
        im = ax.imshow(matrix.to_numpy(), cmap=CMAP_DIVERGING, norm=norm, aspect="auto")

        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                value = matrix.to_numpy()[i, j]
                if np.isnan(value):
                    continue
                ax.text(j, i, f"{value:+.3f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(value) > vmax * 0.55 else "#222222")

        ax.set_xticks(range(len(targets)))
        ax.set_xticklabels(targets, rotation=40, ha="right")
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels(order)
        ax.axhline(len(order) - 1.5, color="black", linewidth=1.2)
        ax.set_title(f"target: {metric_label}", fontsize=10)
        for spine in ax.spines.values():
            spine.set_visible(False)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=r"$\Delta_{i,j}$")

    fig.suptitle(f"{log_label[log]} - per-attribute Delta (excluded feature x target)",
                fontsize=12, y=1.02)
    fig.tight_layout()
    return fig


for log in logs:
    fig = attribute_breakdown_figure(log)
    path = os.path.join(figures_dir, f"03_attribute_breakdown_{log}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path}")

saved /home/danbi/Projects/SANAGRAPH/analysis/figures/03_attribute_breakdown_bpi_2012_CZ.png
saved /home/danbi/Projects/SANAGRAPH/analysis/figures/03_attribute_breakdown_bpi_2013_CZ.png
saved /home/danbi/Projects/SANAGRAPH/analysis/figures/03_attribute_breakdown_sp2020_CZ.png
saved /home/danbi/Projects/SANAGRAPH/analysis/figures/03_attribute_breakdown_BPI20_RequestForPayment_CZ.png


## 4. The single most important feature, across patterns

For the top-ranked feature of each log, `delta_cat` and `delta_num` shown
separately across the five patterns (two different scales, so two columns
rather than one axis - see notebook intro). This is the concrete version of
row 1 of figure 1: not just "org:resource matters for SP 2020" but *when*.

In [13]:
top_feature = (ranking[~ranking.structural]
              .sort_values(["log", "rank"])
              .groupby("log").first().reset_index()[["log", "excluded_feature"]])

fig, axes = plt.subplots(len(logs), 2, figsize=(10, 2.6 * len(logs)))

for row, log in enumerate(logs):
    feature = top_feature.loc[top_feature.log == log, "excluded_feature"].iloc[0]
    d = (delta_aggregated[(delta_aggregated.log == log)
                          & (delta_aggregated.excluded_feature == feature)]
         .set_index("pattern").reindex(patterns))

    for col, metric in enumerate(["delta_cat", "delta_num"]):
        ax = axes[row, col]
        values = d[metric]
        if values.isna().all():
            ax.text(0.5, 0.5, "no surviving numerical target", ha="center", va="center",
                    transform=ax.transAxes, fontsize=9, color=COLOR_AXIS)
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            continue
        x = np.arange(len(patterns))
        ax.bar(x, values, color=sign_color(values.fillna(0)), width=0.6, zorder=3)
        ax.axhline(0, color=COLOR_AXIS, linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(patterns, rotation=30, ha="right")
        label = r"$\Delta_{cat}$ (accuracy)" if metric == "delta_cat" else r"$\Delta_{num}$ (MAE)"
        ax.set_ylabel(label, fontsize=9)
        style_ax(ax, grid_axis="y")

    axes[row, 0].set_title(f"{log_label[log]} - excluded: {feature}", loc="left",
                           fontsize=10, fontweight="bold")

fig.suptitle("Top-ranked feature per log, across missingness patterns", fontsize=13, y=1.005)
fig.tight_layout()

path = os.path.join(figures_dir, "04_top_feature_by_pattern.png")
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"saved {path}")

saved /home/danbi/Projects/SANAGRAPH/analysis/figures/04_top_feature_by_pattern.png


## Recap

In [14]:
for name in sorted(os.listdir(figures_dir)):
    print(f"{name:42s} {os.path.getsize(os.path.join(figures_dir, name)) / 1024:7.1f} KB")

01_ranking_by_log.png                        162.8 KB
02_pattern_heatmap.png                       338.2 KB
03_attribute_breakdown_BPI20_RequestForPayment_CZ.png   219.8 KB
03_attribute_breakdown_bpi_2012_CZ.png       121.1 KB
03_attribute_breakdown_bpi_2013_CZ.png       277.9 KB
03_attribute_breakdown_sp2020_CZ.png          99.4 KB
04_top_feature_by_pattern.png                181.4 KB
